In [1]:
import pandas as pd
# 设定每次读取 10 万行
chunksize = 100000

# 初始化一个计数器或空列表来接收处理后的结果
results = []
behavior_counts = {'pv': 0, 'buy': 0, 'cart': 0, 'fav': 0}
reader = pd.read_csv(r'C:\Users\27144\Downloads\UserBehavior.csv',
                     names=['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp'],
                     chunksize=chunksize)
for chunk in reader:
    # 此时的 chunk 是一个真正的 DataFrame，拥有 .shape 属性
    # 如果你想看当前这一块的行数，可以解开下一行的注释：
    #print(f"正在处理当前分块，行数: {chunk.shape[0]}")
    # 统计当前分块中各种行为的数量
    counts = chunk['behavior_type'].value_counts()
    
    # 将当前分块的结果累加到总计数器中
    for behavior in behavior_counts.keys():
        if behavior in counts:
            behavior_counts[behavior] += counts[behavior]

# 5. 循环结束后，打印最终统计结果
print("最终统计结果：", behavior_counts)


最终统计结果： {'pv': np.int64(89716264), 'buy': np.int64(2015839), 'cart': np.int64(5530446), 'fav': np.int64(2888258)}


In [2]:

total_records = sum(behavior_counts.values())
original_proportions = {k: v / total_records for k, v in behavior_counts.items()}
print("原始数据中各行为类型的比例:")
print(pd.Series(original_proportions))

原始数据中各行为类型的比例:
pv      0.895812
buy     0.020128
cart    0.055221
fav     0.028839
dtype: float64


In [4]:

sample_size = 1_000_000
# 随机种子
random_seed = 42
# 计算各行为类型的抽样比例
sample_ratios = {k: (sample_size * v / total_records) / v
                 for k, v in behavior_counts.items()}

# 第二次遍历：分块进行分层抽样
sample_chunks = []
reader = pd.read_csv(
    r'C:\Users\27144\Downloads\UserBehavior.csv',
    names=['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp'],
    chunksize=chunksize
)

for chunk in reader:
    #  直接用 for 循环遍历分组，完美避开 apply 的所有恶心警告和索引变形
    for behavior, group in chunk.groupby('behavior_type'):
        
        # 安全获取当前行为的抽样比例
        ratio = sample_ratios.get(behavior, 0)
        n_samples = int(len(group) * ratio)
        
        if n_samples > 0:
            # 抽样并将结果直接存入列表
            sampled_group = group.sample(n=n_samples, random_state=random_seed)
            sample_chunks.append(sampled_group)

# 1. 把所有抽样的小块拼成一个大表
sample_df = pd.concat(sample_chunks, ignore_index=True)

# 2. 此时 behavior_type 绝对是老老实实的“普通列”，直接计算比例，100% 不会报错！
sampled_proportions = sample_df['behavior_type'].value_counts(normalize=True)
print("\n抽样后数据中各行为类型的比例:")
print(sampled_proportions)

# 3. 将抽样后的数据存储到本地
sample_df.to_csv(r'C:\Users\27144\test\sampled_data.csv', index=False)


抽样后数据中各行为类型的比例:
behavior_type
pv      0.897110
cart    0.054831
fav     0.028395
buy     0.019664
Name: proportion, dtype: float64


In [8]:


file_path =r'C:\Users\27144\test\sampled_data.csv'
sampled_df = pd.read_csv(file_path)

# 数据预览
print("数据基本信息：")
sampled_df.info()

# 查看数据集行数和列数
rows, columns = sampled_df.shape

if rows > 0:
    # 数据行数大于 0 才进行后续操作
    #把 sample_df 这个数据集的前 5 行数据，转换成一种用“制表符（Tab键）”分隔的字符串格式，并把里面的缺失值（空值）显式地替换为 'nan' 字符串，最后打印在屏幕上。
    print("\n数据前几行信息：")
    print(sampled_df.head().to_csv(sep='\t', na_rep='nan'))

    # 查看各列的缺失值情况
    #Pandas中的isnull()函数用于检测DataFrame中的缺失值，并返回一个布尔型DataFrame，缺失值的位置标记为True，其他值标记为False。
    missing_values = sampled_df.isnull().sum()
    print("\n各列缺失值数量：")
    print(missing_values)

    # 处理缺失值（这里简单地删除含有缺失值的行）
    #pandas的dropna函数用于删除DataFrame或Series中包含缺失值（NaN）的行或列，以便清理数据。
    if missing_values.sum() > 0:
        sampled_df = sampled_df.dropna()
        print("\n已删除含有缺失值的行。")

    # 查看重复值数量
    duplicate_count = sampled_df.duplicated().sum()
    print(f"\n重复值数量：{duplicate_count}")

    # 处理重复值
    if duplicate_count > 0:
        sampled_df = sampled_df.drop_duplicates()
        print("已删除重复值。")

    # 处理异常值（以 timestamp 列为例，假设时间戳应在合理范围内）
    if 'timestamp' in sampled_df.columns:
        # 先将时间戳转换为日期时间类型
        sampled_df['timestamp'] = pd.to_datetime(sampled_df['timestamp'], unit='s')
        # 假设数据时间范围在 2017 年 11 月 25 日至 2017 年 12 月 3 日
        start_date = pd.Timestamp('2017-11-25')
        end_date = pd.Timestamp('2017-12-03')
        outlier_mask = (sampled_df['timestamp'] < start_date) | (sampled_df['timestamp'] > end_date)
        outlier_count = outlier_mask.sum()
        print(f"\ntimestamp 列异常值数量：{outlier_count}")

        # 处理异常值（删除异常值）
        if outlier_count > 0:
            sampled_df = sampled_df[~outlier_mask]
            print("已删除 timestamp 列的异常值。")

    # 将清洗后的数据存储到本地 D:/Desktop
    cleaned_file_path = r'C:\Users\27144\test\cleaned_sampled_data.csv'
    sampled_df.to_csv(cleaned_file_path, index=False)
    print(f"\n清洗后的数据已保存至 {cleaned_file_path}")
else:
    print("抽样数据文件为空，无法进行后续操作。")

数据基本信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 997988 entries, 0 to 997987
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   user_id        997988 non-null  int64 
 1   item_id        997988 non-null  int64 
 2   category_id    997988 non-null  int64 
 3   behavior_type  997988 non-null  object
 4   timestamp      997988 non-null  int64 
dtypes: int64(4), object(1)
memory usage: 38.1+ MB

数据前几行信息：
	user_id	item_id	category_id	behavior_type	timestamp
0	1004337	2187255	3439012	buy	1511767626
1	1002413	4638382	1464116	buy	1511704155
2	1002306	3988018	1090997	buy	1511940738
3	100234	1641665	3901339	buy	1511857679
4	1003983	3050742	3740459	buy	1511998190


各列缺失值数量：
user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
dtype: int64

重复值数量：0

timestamp 列异常值数量：131377
已删除 timestamp 列的异常值。

清洗后的数据已保存至 C:\Users\27144\test\cleaned_sampled_data.csv
